In [1]:
import numpy as np
from statsmodels.tsa.statespace.dynamic_factor import DynamicFactor
from pymc_extras.statespace.models.DFM import BayesianDynamicFactor

import pytensor

from pymc_extras.statespace.utils.constants import (
    MATRIX_NAMES,
    SHORT_NAME_TO_LONG,
)
def unpack_statespace(ssm):
    return [ssm[SHORT_NAME_TO_LONG[x]] for x in MATRIX_NAMES]

In [2]:
# Set the same seed for both
seed = 42

# For statsmodels
np.random.seed(seed) 

# For PyMC-extras
rng = np.random.default_rng(seed)

In [3]:
# Example parameters
n_obs = 100
endog = np.random.normal(size=(n_obs, 2))  
exog = np.random.normal(size=(n_obs, 2)) 

In [4]:
# Create DFM Statespace Model
dfm_mod = BayesianDynamicFactor(
    k_factors=1,
    factor_order=1,
    k_endog=2,
    error_order=1,
    error_var=False,
    k_exog=2,
    shared_exog_states=False,
    exog_innovations=True,
    error_cov_type="diagonal",
    measurement_error=False,
    verbose=True
)

                                     Model Requirements                                      
                                                                                             
  Variable          Shape       Constraints                                      Dimensions  
 ─────────────────────────────────────────────────────────────────────────────────────────── 
  x0                (7,)                                                         ('state',)  
  P0                (7, 7)      Positive Semi-definite               ('state', 'state_aux')  
  factor_loadings   (2, 1)                                     ('observed_state', 'factor')  
  factor_ar         (1, 1)                                             ('factor', 'lag_ar')  
  error_ar          (2, 1)                               ('observed_state', 'error_lag_ar')  
  error_sigma       (2,)        Positive                                ('observed_state',)  
  beta              (4,)                                               ('exogenous_state',)  
  beta_sigma        (4,)        Positive                               ('exogenous_state',)  
                                                                                             
  exog_data         (None, 2)   pm.Data                         ('time', 'exogenous_state')  
                                                                                             
   These parameters should be assigned priors inside a PyMC model block before calling the   
                               build_statespace_graph method.                                

In [13]:
# Create parameter dictionary matching only the expected parameters
expected_params = list(dfm_mod._name_to_variable.keys())
expected_data = list(dfm_mod._name_to_data.keys())

print("Expected parameters:", expected_params)
print("Expected data:", expected_data)

# Build param_dict with only the parameters the model expects
param_dict = {}
data_dict = {}

# Add parameters based on what's actually expected
if "factor_loadings" in expected_params:
    param_dict["factor_loadings"] = np.array([[0.9], [0.8]])
if "factor_ar" in expected_params:
    param_dict["factor_ar"] = np.array([[0.5]])
if "error_ar" in expected_params:
    param_dict["error_ar"] = np.array([[0.4], [0.3]])
if "error_sigma" in expected_params:
    param_dict["error_sigma"] = np.array([np.sqrt(0.7), np.sqrt(0.6)])
if "P0" in expected_params:
    param_dict["P0"] = np.eye(dfm_mod.k_states)
if "x0" in expected_params:
    param_dict["x0"] = np.array([0.0, 0.0, 0.0])  # Initial state
if "beta" in expected_params:
    param_dict["beta"] = np.array([0.3, 0.5, 1, 2])
if "beta_sigma" in expected_params:
    param_dict["beta_sigma"] = np.array([np.sqrt(1), np.sqrt(2), np.sqrt(3), np.sqrt(4)])

# Add data based on what's expected
if "exog_data" in expected_data:
    data_dict["exog_data"] = exog

Expected parameters: ['x0', 'beta', 'P0', 'factor_loadings', 'factor_ar', 'error_ar', 'error_sigma', 'beta_sigma']
Expected data: ['exog_data']


In [14]:
print("Final param_dict keys:", list(param_dict.keys()))
print("Final data_dict keys:", list(data_dict.keys()))

Final param_dict keys: ['factor_loadings', 'factor_ar', 'error_ar', 'error_sigma', 'P0', 'x0', 'beta', 'beta_sigma']
Final data_dict keys: ['exog_data']


In [15]:
def unpack_symbolic_matrices_with_params(mod, param_dict, data_dict=None, mode="FAST_COMPILE"):
    inputs = list(mod._name_to_variable.values())
    if data_dict is not None:
        inputs += list(mod._name_to_data.values())
    else:
        data_dict = {}

    f_matrices = pytensor.function(
        inputs,
        unpack_statespace(mod.ssm),
        on_unused_input="raise",
        mode=mode,
    )

    x0, P0, c, d, T, Z, R, H, Q = f_matrices(**param_dict, **data_dict)

    return x0, P0, c, d, T, Z, R, H, Q


def simulate_from_numpy_model(mod, rng, param_dict, data_dict=None, steps=100, state_shocks=None, measurement_shocks=None):
    """
    Helper function to visualize the components outside of a PyMC model context

    Modified version in order to take as input state_shocks and measurement_shocks otherwise 
    the random quantities between stats and pymc will be different leading to different obs variables
    """
    x0, P0, c, d, T, Z, R, H, Q = unpack_symbolic_matrices_with_params(mod, param_dict, data_dict)
    k_endog = mod.k_endog
    k_states = mod.k_states
    k_posdef = mod.k_posdef

    x = np.zeros((steps, k_states))
    y = np.zeros((steps, k_endog))

    x[0] = x0
    y[0] = (Z @ x0).squeeze() if Z.ndim == 2 else (Z[0] @ x0).squeeze()

    if not np.allclose(H, 0):
        y[0] += rng.multivariate_normal(mean=np.zeros(1), cov=H).squeeze()

    for t in range(1, steps):
        if k_posdef > 0:
            innov = R @ rng.multivariate_normal(mean=np.zeros(k_posdef), cov=Q)
            #shock = np.concatenate((state_shocks[t - 1], np.zeros(4)), axis=0)
            #innov = R @ shock
        else:
            innov = 0

        if not np.allclose(H, 0):
           error = measurement_shocks[t - 1]
        else:
            error = 0

        x[t] = c + T @ x[t - 1] + innov
        if Z.ndim == 2:
            y[t] = (d + Z @ x[t] + error).squeeze()
        else:
            y[t] = (d + Z[t] @ x[t] + error).squeeze()

    return x, y.squeeze()

In [16]:
# Simulate trajectories
rng = np.random.default_rng(123)
x_traj, y_traj = simulate_from_numpy_model(
    dfm_mod, rng, param_dict, data_dict, steps=n_obs
)

In [ ]:
# --- Step 2: Identify where betas live in the state vector ---
# If shared_exog_states=False:
#   Each endog has its own set of betas
# So total exog states = k_exog * k_endog
k_exog_states = dfm_mod.k_exog * dfm_mod.k_endog
beta_traj = x_traj[:, -k_exog_states:]   # slice the last block as betas

# --- Step 3: Compute variance over simulation runs ---
# If you want multiple runs, simulate many times
n_runs = 500
betas_t1 = []
betas_t100 = []

for _ in range(n_runs):
    x_traj, _ = simulate_from_numpy_model(dfm_mod, rng, param_dict, data_dict, steps=n_obs)
    beta_traj = x_traj[:, -k_exog_states:]
    betas_t1.append(beta_traj[1, :])
    betas_t100.append(beta_traj[99, :])

betas_t1 = np.array(betas_t1)
betas_t100 = np.array(betas_t100)

# --- Step 4: Compare variances ---
var_t1 = betas_t1.var(axis=0)
var_t100 = betas_t100.var(axis=0)

print("Variance at T=1:", var_t1)
print("Variance at T=100:", var_t100)
print("Check (T=100 > T=1):", np.all(var_t100 > var_t1))


MissingInputError: Input 0 (exog_data) of the graph (indices start from 0), used to compute ExpandDims{axis=1}(exog_data), was not provided and not given a value. Use the PyTensor flag exception_verbosity='high', for more information on this error.
 
Backtrace when that variable is created:

  File "/opt/anaconda3/envs/pymc_extras/lib/python3.11/site-packages/IPython/core/interactiveshell.py", line 3394, in run_cell_async
    has_raised = await self.run_ast_nodes(code_ast.body, cell_name,
  File "/opt/anaconda3/envs/pymc_extras/lib/python3.11/site-packages/IPython/core/interactiveshell.py", line 3639, in run_ast_nodes
    if await self.run_code(code, result, async_=asy):
  File "/opt/anaconda3/envs/pymc_extras/lib/python3.11/site-packages/IPython/core/interactiveshell.py", line 3699, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/var/folders/kx/1mlzw51d67d3cr090d4kd9zc0000gn/T/ipykernel_60068/3416084277.py", line 2, in <module>
    dfm_mod = BayesianDynamicF